In [3]:
import os
from collections import Counter

# Set the path to t/home/ai/Documents/ppe data/labelshe YOLO dataset labels directory
labels_dir =  "/home/variphi/gowtham/model_training/cumi/05-06-26_split/labels/train" # adjust if your labels are elsewhere

class_counts = Counter()

for label_file in os.listdir(labels_dir):
    if not label_file.endswith('.txt'):
        continue
    with open(os.path.join(labels_dir, label_file), 'r') as f:
        for line in f:
            if line.strip() == "":
                continue
            class_id = line.strip().split()[0]
            class_counts[class_id] += 1

print("Number of instances per class (by class id):")
for class_id, count in sorted(class_counts.items(), key=lambda x: int(float(x[0]))):
    print(f"Class {int(float(class_id))}: {count}")

Number of instances per class (by class id):
Class 0: 10340
Class 1: 4309
Class 2: 2403
Class 3: 1369
Class 4: 1310
Class 5: 1204
Class 6: 6771
Class 7: 2380
Class 8: 5455
Class 9: 1828
Class 10: 1928


seperation_class

In [ ]:
import os
import shutil
from tqdm import tqdm

# =====================================================
# CONFIG
# =====================================================

IMAGE_DIR = "/home/variphi/gowtham/model_training/cumi/images/"
LABEL_DIR = "/home/variphi/gowtham/model_training/cumi/labels/"

TARGET_CLASS_ID = 7
CLASS_NAME = "apron"

OUTPUT_DIR = "/home/variphi/gowtham/model_training/cumi/apron"

# =====================================================
# OUTPUT FOLDERS
# =====================================================

OUTPUT_IMAGES = os.path.join(OUTPUT_DIR, "images")
OUTPUT_LABELS = os.path.join(OUTPUT_DIR, "labels")

os.makedirs(OUTPUT_IMAGES, exist_ok=True)
os.makedirs(OUTPUT_LABELS, exist_ok=True)

# =====================================================
# PROCESS
# =====================================================

count = 0

for label_file in tqdm(os.listdir(LABEL_DIR)):

    if not label_file.endswith(".txt"):
        continue

    label_path = os.path.join(LABEL_DIR, label_file)

    with open(label_path, "r") as f:
        lines = f.readlines()

    found = False

    for line in lines:
        parts = line.strip().split()

        if len(parts) < 5:
            continue

        if int(parts[0]) == TARGET_CLASS_ID:
            found = True
            break

    if not found:
        continue

    base_name = os.path.splitext(label_file)[0]

    image_path = None

    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        temp_path = os.path.join(IMAGE_DIR, base_name + ext)

        if os.path.exists(temp_path):
            image_path = temp_path
            break

    # Move image
    if image_path:
        shutil.move(
            image_path,
            os.path.join(
                OUTPUT_IMAGES,
                os.path.basename(image_path)
            )
        )

    # Move label
    shutil.move(
        label_path,
        os.path.join(
            OUTPUT_LABELS,
            label_file
        )
    )

    count += 1

print(f"\nMoved {count} image-label pairs containing class {TARGET_CLASS_ID}")

Processing /home/variphi/gowtham/model_training/cumi/05-06-26_split/labels/train/: 100%|██████████| 12276/12276 [00:04<00:00, 2669.31it/s]
Processing /home/variphi/gowtham/model_training/cumi/05-06-26_split/labels/val/: 100%|██████████| 3070/3070 [00:01<00:00, 2722.47it/s]


Target Class ID : 7
Total Files Copied : 2780
Saved To : /home/variphi/gowtham/model_training/cumi/apron


copy paste argumentation

In [20]:
import os
import cv2
import numpy as np
import random
from collections import defaultdict
from tqdm import tqdm

# =========================================================
# CONFIGURATION
# =========================================================

# Dataset paths
IMAGE_DIR = "/home/variphi/gowtham/model_training/cumi/part_2/part_2/images"
LABEL_DIR = "/home/variphi/gowtham/model_training/cumi/part_2/part_2/labels"

# Output paths
OUTPUT_IMAGE_DIR = "/home/variphi/gowtham/model_training/cumi/part_2/part_2/aug_images"
OUTPUT_LABEL_DIR = "/home/variphi/gowtham/model_training/cumi/part_2/part_2/aug_labels"

# Classes to augment
CLASSES_TO_AUGMENT = [7]

# Target instance count
TARGET_INSTANCE_COUNT = 1500

# Augmentation parameters
IOU_THRESHOLD = 0.2
TARGET_SIZE_RATIO = 0.02

# Bottom placement control
BOTTOM_REGION_START = 0.60   # lower 40% of image
BOTTOM_MARGIN = 10


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def yolo_to_pixel(yolo_box, img_width, img_height):
    x_center, y_center, w, h = yolo_box

    x_center *= img_width
    y_center *= img_height
    w *= img_width
    h *= img_height

    x_min = int(x_center - w / 2)
    y_min = int(y_center - h / 2)
    x_max = int(x_center + w / 2)
    y_max = int(y_center + h / 2)

    return x_min, y_min, x_max, y_max


def pixel_to_yolo(pixel_box, img_width, img_height):
    x_min, y_min, x_max, y_max = pixel_box

    w = x_max - x_min
    h = y_max - y_min

    x_center = x_min + w / 2
    y_center = y_min + h / 2

    x_center /= img_width
    y_center /= img_height
    w /= img_width
    h /= img_height

    return x_center, y_center, w, h


def calculate_iou(box1, box2):

    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])

    inter_area = max(0, x2_inter - x1_inter) * max(0, y2_inter - y1_inter)

    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union_area = box1_area + box2_area - inter_area

    if union_area <= 0:
        return 0

    return inter_area / union_area


# =========================================================
# PREPARE DATASET ASSETS
# =========================================================

def prepare_dataset_assets(label_dir, image_dir):

    print("Scanning dataset and preparing assets...")

    all_background_paths = [
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    class_instances = defaultdict(list)
    initial_class_counts = defaultdict(int)

    label_files = [
        f for f in os.listdir(label_dir)
        if f.endswith(".txt")
    ]

    for label_file in tqdm(
        label_files,
        desc="Loading annotations"
    ):

        label_path = os.path.join(label_dir, label_file)

        with open(label_path, "r") as f:
            lines = f.readlines()

        base_name = os.path.splitext(label_file)[0]

        img_ext = next(
            (
                ext for ext in [".jpg", ".jpeg", ".png"]
                if os.path.exists(
                    os.path.join(
                        image_dir,
                        base_name + ext
                    )
                )
            ),
            None
        )

        if img_ext is None:
            continue

        image_path = os.path.join(
            image_dir,
            base_name + img_ext
        )

        for line in lines:

            parts = line.strip().split()

            if not parts:
                continue

            class_id = int(parts[0])
            bbox = [float(x) for x in parts[1:]]

            class_instances[class_id].append(
                {
                    "image_path": image_path,
                    "bbox": bbox
                }
            )

            initial_class_counts[class_id] += 1

    print("Asset preparation complete.")

    return (
        class_instances,
        all_background_paths,
        initial_class_counts
    )


# =========================================================
# COPY PASTE AUGMENTATION
# =========================================================

def copy_paste_augment(
    image_dir,
    label_dir,
    output_image_dir,
    output_label_dir,
    classes_to_augment,
    target_instance_count,
    iou_threshold,
    target_size_ratio
):

    print("Starting copy-paste augmentation...")

    if not os.path.isdir(image_dir):
        print(f"❌ Image directory not found: {image_dir}")
        return

    if not os.path.isdir(label_dir):
        print(f"❌ Label directory not found: {label_dir}")
        return

    os.makedirs(output_image_dir, exist_ok=True)
    os.makedirs(output_label_dir, exist_ok=True)

    (
        class_instances,
        all_background_paths,
        initial_class_counts
    ) = prepare_dataset_assets(
        label_dir,
        image_dir
    )

    if not all_background_paths:
        print("❌ No images found.")
        return

    print("\n--- Initial Class Counts ---")

    for cid in sorted(initial_class_counts.keys()):
        print(
            f"Class ID {cid}: "
            f"{initial_class_counts[cid]}"
        )

    print("-" * 30)

    for class_id in classes_to_augment:

        print(f"\nAugmenting class: {class_id}")

        source_objects = class_instances.get(
            class_id,
            []
        )

        if not source_objects:
            print(
                f"⚠️ No instances found "
                f"for class {class_id}"
            )
            continue

        current_count = initial_class_counts.get(
            class_id,
            0
        )

        augmentations_needed = (
            target_instance_count -
            current_count
        )

        if augmentations_needed <= 0:
            print(
                f"✅ Class {class_id} "
                f"already has "
                f"{current_count} instances"
            )
            continue

        print(
            f"Current count: "
            f"{current_count}"
        )

        print(
            f"Generating "
            f"{augmentations_needed} "
            f"new samples..."
        )

        pbar = tqdm(
            range(augmentations_needed),
            desc=f"Class {class_id}"
        )

        for i in pbar:

            try:

                # ==================================
                # 1. SELECT SOURCE & DESTINATION
                # ==================================

                source_obj = random.choice(
                    source_objects
                )

                dest_image_path = random.choice(
                    all_background_paths
                )

                # ==================================
                # 2. LOAD SOURCE IMAGE
                # ==================================

                source_image = cv2.imread(
                    source_obj["image_path"]
                )

                if source_image is None:
                    continue

                src_h, src_w = (
                    source_image.shape[:2]
                )

                src_xmin, src_ymin, src_xmax, src_ymax = (
                    yolo_to_pixel(
                        source_obj["bbox"],
                        src_w,
                        src_h
                    )
                )

                cropped_object = source_image[
                    src_ymin:src_ymax,
                    src_xmin:src_xmax
                ]

                if cropped_object.size == 0:
                    continue

                crop_h, crop_w = (
                    cropped_object.shape[:2]
                )

                # ==================================
                # 3. LOAD DESTINATION IMAGE
                # ==================================

                dest_image = cv2.imread(
                    dest_image_path
                )

                if dest_image is None:
                    continue

                dest_h, dest_w = (
                    dest_image.shape[:2]
                )

                # ==================================
                # 4. RESIZE OBJECT
                # ==================================

                dest_area = (
                    dest_h * dest_w
                )

                target_crop_area = (
                    dest_area *
                    target_size_ratio
                )

                scale_factor = np.sqrt(
                    target_crop_area /
                    (crop_h * crop_w)
                )

                new_w = int(
                    crop_w *
                    scale_factor
                )

                new_h = int(
                    crop_h *
                    scale_factor
                )

                if new_w >= dest_w:
                    new_w = dest_w - 1

                if new_h >= dest_h:
                    new_h = dest_h - 1

                if (
                    new_w <= 0 or
                    new_h <= 0
                ):
                    continue

                resized_object = cv2.resize(
                    cropped_object,
                    (new_w, new_h),
                    interpolation=cv2.INTER_AREA
                )

                crop_w = new_w
                crop_h = new_h

                # ==================================
                # 5. PASTE AT LOWER 40%
                # ==================================

                paste_x = random.randint(
                    0,
                    dest_w - crop_w
                )

                min_y = int(
                    dest_h *
                    BOTTOM_REGION_START
                )

                max_y = (
                    dest_h -
                    crop_h -
                    BOTTOM_MARGIN
                )

                if max_y <= min_y:
                    continue

                paste_y = random.randint(
                    min_y,
                    max_y
                )

                dest_image[
                    paste_y:paste_y + crop_h,
                    paste_x:paste_x + crop_w
                ] = resized_object

                # ==================================
                # 6. HANDLE LABELS
                # ==================================

                new_pixel_box = (
                    paste_x,
                    paste_y,
                    paste_x + crop_w,
                    paste_y + crop_h
                )

                dest_base_name = (
                    os.path.splitext(
                        os.path.basename(
                            dest_image_path
                        )
                    )[0]
                )

                original_label_path = os.path.join(
                    label_dir,
                    f"{dest_base_name}.txt"
                )

                final_labels = []

                if os.path.exists(
                    original_label_path
                ):

                    with open(
                        original_label_path,
                        "r"
                    ) as f:

                        for line in f.readlines():

                            parts = (
                                line.strip()
                                .split()
                            )

                            if not parts:
                                continue

                            orig_bbox = [
                                float(x)
                                for x in parts[1:]
                            ]

                            orig_pixel_box = (
                                yolo_to_pixel(
                                    orig_bbox,
                                    dest_w,
                                    dest_h
                                )
                            )

                            iou = calculate_iou(
                                new_pixel_box,
                                orig_pixel_box
                            )

                            if (
                                iou <
                                iou_threshold
                            ):
                                final_labels.append(
                                    line.strip()
                                )

                # Add new label
                new_yolo_box = pixel_to_yolo(
                    new_pixel_box,
                    dest_w,
                    dest_h
                )

                final_labels.append(
                    f"{class_id} "
                    f"{' '.join(map(str, new_yolo_box))}"
                )

                # ==================================
                # 7. SAVE
                # ==================================

                filename = (
                    f"{dest_base_name}"
                    f"_aug_{class_id}_{i}"
                )

                output_image_path = os.path.join(
                    output_image_dir,
                    filename + ".jpg"
                )

                output_label_path = os.path.join(
                    output_label_dir,
                    filename + ".txt"
                )

                cv2.imwrite(
                    output_image_path,
                    dest_image
                )

                with open(
                    output_label_path,
                    "w"
                ) as f:

                    f.write(
                        "\n".join(
                            final_labels
                        )
                    )

            except Exception as e:
                print(f"\n⚠️ Error: {e}")
                continue

    print("\n🎉 Augmentation completed.")
    print(
        f"Images saved to: "
        f"{output_image_dir}"
    )
    print(
        f"Labels saved to: "
        f"{output_label_dir}"
    )


# =========================================================
# RUN
# =========================================================

if __name__ == "__main__":

    copy_paste_augment(
        image_dir=IMAGE_DIR,
        label_dir=LABEL_DIR,
        output_image_dir=OUTPUT_IMAGE_DIR,
        output_label_dir=OUTPUT_LABEL_DIR,
        classes_to_augment=CLASSES_TO_AUGMENT,
        target_instance_count=TARGET_INSTANCE_COUNT,
        iou_threshold=IOU_THRESHOLD,
        target_size_ratio=TARGET_SIZE_RATIO
    )

Starting copy-paste augmentation...
Scanning dataset and preparing assets...


Loading annotations: 100%|██████████| 4559/4559 [00:00<00:00, 36432.25it/s]


Asset preparation complete.

--- Initial Class Counts ---
Class ID 0: 3690
Class ID 1: 1542
Class ID 2: 291
Class ID 3: 169
Class ID 4: 124
Class ID 5: 7
Class ID 6: 2144
Class ID 7: 839
Class ID 8: 2886
------------------------------

Augmenting class: 7
Current count: 839
Generating 661 new samples...


Class 7: 100%|██████████| 661/661 [00:21<00:00, 30.93it/s]


🎉 Augmentation completed.
Images saved to: /home/variphi/gowtham/model_training/cumi/part_2/part_2/aug_images
Labels saved to: /home/variphi/gowtham/model_training/cumi/part_2/part_2/aug_labels


In [7]:
import os
import cv2
import random
from tqdm import tqdm

# =====================================================
# CONFIG
# =====================================================

IMAGE_DIR = "/home/variphi/gowtham/model_training/cumi/part_2/part_2/cumi/images"
LABEL_DIR = "/home/variphi/gowtham/model_training/cumi/part_2/part_2/cumi/labels"

BACKGROUND_DIR = "/home/variphi/background"

OUTPUT_IMAGE_DIR = "/home/variphi/gowtham/model_training/cumi/part_2/part_2/cumi/cumi_aug/images"
OUTPUT_LABEL_DIR = "/home/variphi/gowtham/model_training/cumi/part_2/part_2/cumi/cumi_aug/labels"

TARGET_CLASS = 7
NUM_AUG_IMAGES = 2800

# Object occupies ~25% of frame width
TARGET_WIDTH_RATIO = 0.25

# =====================================================
# YOLO <-> PIXEL
# =====================================================

def yolo_to_pixel(box, img_w, img_h):

    x, y, w, h = box

    x *= img_w
    y *= img_h
    w *= img_w
    h *= img_h

    xmin = int(x - w / 2)
    ymin = int(y - h / 2)
    xmax = int(x + w / 2)
    ymax = int(y + h / 2)

    return xmin, ymin, xmax, ymax


def pixel_to_yolo(box, img_w, img_h):

    xmin, ymin, xmax, ymax = box

    bw = xmax - xmin
    bh = ymax - ymin

    xc = xmin + bw / 2
    yc = ymin + bh / 2

    return (
        xc / img_w,
        yc / img_h,
        bw / img_w,
        bh / img_h
    )

# =====================================================
# COLLECT OBJECTS
# =====================================================

def collect_objects():

    objects = []

    label_files = [
        f for f in os.listdir(LABEL_DIR)
        if f.endswith(".txt")
    ]

    print(f"Found {len(label_files)} label files")

    for label_file in tqdm(label_files):

        base_name = os.path.splitext(label_file)[0]

        image_path = None

        for ext in [".jpg", ".jpeg", ".png"]:

            temp_path = os.path.join(
                IMAGE_DIR,
                base_name + ext
            )

            if os.path.exists(temp_path):
                image_path = temp_path
                break

        if image_path is None:
            continue

        image = cv2.imread(image_path)

        if image is None:
            continue

        img_h, img_w = image.shape[:2]

        label_path = os.path.join(
            LABEL_DIR,
            label_file
        )

        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            cls = int(parts[0])

            if cls != TARGET_CLASS:
                continue

            bbox = list(
                map(float, parts[1:])
            )

            xmin, ymin, xmax, ymax = (
                yolo_to_pixel(
                    bbox,
                    img_w,
                    img_h
                )
            )

            xmin = max(0, xmin)
            ymin = max(0, ymin)
            xmax = min(img_w, xmax)
            ymax = min(img_h, ymax)

            crop = image[
                ymin:ymax,
                xmin:xmax
            ]

            if crop.size == 0:
                continue

            objects.append(crop)

    return objects

# =====================================================
# GENERATE
# =====================================================

def generate():

    os.makedirs(
        OUTPUT_IMAGE_DIR,
        exist_ok=True
    )

    os.makedirs(
        OUTPUT_LABEL_DIR,
        exist_ok=True
    )

    objects = collect_objects()

    if len(objects) == 0:
        print("No objects found.")
        return

    backgrounds = [
        os.path.join(BACKGROUND_DIR, f)
        for f in os.listdir(BACKGROUND_DIR)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

    if len(backgrounds) == 0:
        print("No backgrounds found.")
        return

    print(f"Objects collected: {len(objects)}")
    print(f"Background images: {len(backgrounds)}")

    for idx in tqdm(
        range(NUM_AUG_IMAGES),
        desc="Generating"
    ):

        bg_path = random.choice(backgrounds)

        background = cv2.imread(bg_path)

        if background is None:
            continue

        bg_h, bg_w = background.shape[:2]

        obj = random.choice(objects)

        obj_h, obj_w = obj.shape[:2]

        # =====================================
        # AUTO SCALE OBJECT
        # =====================================

        target_width = int(
            bg_w * TARGET_WIDTH_RATIO
        )

        scale = target_width / obj_w

        new_w = int(obj_w * scale)
        new_h = int(obj_h * scale)

        if new_w <= 0 or new_h <= 0:
            continue

        if new_w >= bg_w:
            continue

        if new_h >= bg_h:
            continue

        obj = cv2.resize(
            obj,
            (new_w, new_h),
            interpolation=cv2.INTER_AREA
        )

        # =====================================
        # CENTER POSITION
        # =====================================

        x = (bg_w - new_w) // 2
        y = (bg_h - new_h) // 2

        # =====================================
        # PASTE OBJECT
        # =====================================

        background[
            y:y + new_h,
            x:x + new_w
        ] = obj

        # =====================================
        # LABEL
        # =====================================

        bbox = (
            x,
            y,
            x + new_w,
            y + new_h
        )

        yolo_box = pixel_to_yolo(
            bbox,
            bg_w,
            bg_h
        )

        image_name = f"aug_{idx:06d}.jpg"
        label_name = f"aug_{idx:06d}.txt"

        cv2.imwrite(
            os.path.join(
                OUTPUT_IMAGE_DIR,
                image_name
            ),
            background
        )

        with open(
            os.path.join(
                OUTPUT_LABEL_DIR,
                label_name
            ),
            "w"
        ) as f:

            f.write(
                f"{TARGET_CLASS} "
                f"{yolo_box[0]:.6f} "
                f"{yolo_box[1]:.6f} "
                f"{yolo_box[2]:.6f} "
                f"{yolo_box[3]:.6f}"
            )

    print("\nDone!")
    print(f"Images: {OUTPUT_IMAGE_DIR}")
    print(f"Labels: {OUTPUT_LABEL_DIR}")

# =====================================================
# RUN
# =====================================================

if __name__ == "__main__":
    generate()

Found 13546 label files


100%|██████████| 13546/13546 [02:08<00:00, 105.04it/s]


Objects collected: 2252
Background images: 1


Generating: 100%|██████████| 2800/2800 [03:15<00:00, 14.33it/s]


Done!
Images: /home/variphi/gowtham/model_training/cumi/part_2/part_2/cumi/cumi_aug/images
Labels: /home/variphi/gowtham/model_training/cumi/part_2/part_2/cumi/cumi_aug/labels


In [19]:
pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

# =========================================================
# CONFIGURATION
# =========================================================

LABEL_DIR = "/home/variphi/gowtham/model_training/cumi/labels/"

# Enter class ID to remove
REMOVE_CLASS_ID = int(input("Enter class ID to remove: "))

# =========================================================
# GET LABEL FILES
# =========================================================

label_files = [
    f for f in os.listdir(LABEL_DIR)
    if f.endswith(".txt")
]

print(f"\nTotal Label Files Found: {len(label_files)}")

modified_files = 0
removed_objects = 0

# =========================================================
# PROCESS LABEL FILES
# =========================================================

for label_file in label_files:

    label_path = os.path.join(LABEL_DIR, label_file)

    with open(label_path, "r") as f:
        lines = f.readlines()

    updated_lines = []

    for line in lines:

        parts = line.strip().split()

        # Skip invalid lines
        if len(parts) < 5:
            continue

        class_id = int(parts[0])

        # Remove selected class
        if class_id == REMOVE_CLASS_ID:
            removed_objects += 1
            continue

        updated_lines.append(line)

    # Rewrite updated labels
    with open(label_path, "w") as f:
        f.writelines(updated_lines)

    # Count modified files
    if len(updated_lines) != len(lines):
        modified_files += 1

# =========================================================
# SUMMARY
# =========================================================

print("\n===================================")
print("CLASS REMOVAL COMPLETED")
print("===================================")

print(f"Removed Class ID : {REMOVE_CLASS_ID}")
print(f"Modified Files   : {modified_files}")
print(f"Removed Objects  : {removed_objects}")


Total Label Files Found: 2903

CLASS REMOVAL COMPLETED
Removed Class ID : 10
Modified Files   : 3
Removed Objects  : 3


In [ ]:
import os

def switch_multiple_class_ids_in_labels(labels_dir, id_map):
    """
    Switches multiple class ids in all YOLO label files in the given directory.
    Args:
        labels_dir (str): Path to the directory containing YOLO label .txt files.
        id_map (dict): Dictionary mapping old class ids (as int or str) to new class ids (as int or str).
    """
    # Convert all keys and values in id_map to strings for comparison and replacement
    id_map_str = {str(k): str(v) for k, v in id_map.items()}
    for filename in os.listdir(labels_dir):
        if filename.endswith('.txt'):
            file_path = os.path.join(labels_dir, filename)
            with open(file_path, 'r') as f:
                lines = f.readlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) > 0 and parts[0] in id_map_str:
                    parts[0] = id_map_str[parts[0]]
                new_lines.append(' '.join(parts) + '\n')
            with open(file_path, 'w') as f:
                f.writelines(new_lines)

# Example usage:/home/ai/Documents/YOLOv8Training/Base_Dataset/MergedData (copy)/labels/
switch_multiple_class_ids_in_labels('/home/variphi/sam3/output/labels_1', { 7 : 9, 1 : 10 })

In [ ]:
import os
import random
import shutil
import sys

def split_yolo_dataset(source_dir, output_dir, val_split_ratio=0.2):
    """
    Splits a YOLO format dataset into training and validation sets.

    The function creates a new directory structure and copies the files,
    ensuring that for every image in a set, its corresponding label exists.

    Args:
        source_dir (str): Path to the source dataset directory which contains
                          'images' and 'labels' subdirectories.
        output_dir (str): Path to the directory where the split dataset will
                          be saved.
        val_split_ratio (float): The proportion of the dataset to allocate
                                 to the validation set.
    """
    print("--- Starting YOLO Dataset Split ---")

    # --- 1. Define Source and Destination Paths ---
    source_images_dir = os.path.join(source_dir, 'images')
    source_labels_dir = os.path.join(source_dir, 'labels')

    if not os.path.isdir(source_images_dir):
        print(f"[ERROR] 'images' directory not found in '{source_dir}'")
        sys.exit(1)
    if not os.path.isdir(source_labels_dir):
        print(f"[ERROR] 'labels' directory not found in '{source_dir}'")
        sys.exit(1)

    train_images_dir = os.path.join(output_dir, 'images', 'train')
    val_images_dir = os.path.join(output_dir, 'images', 'val')
    train_labels_dir = os.path.join(output_dir, 'labels', 'train')
    val_labels_dir = os.path.join(output_dir, 'labels', 'val')

    # --- 2. Create Output Directories ---
    print(f"Creating output directories in '{output_dir}'...")
    os.makedirs(train_images_dir, exist_ok=True)
    os.makedirs(val_images_dir, exist_ok=True)
    os.makedirs(train_labels_dir, exist_ok=True)
    os.makedirs(val_labels_dir, exist_ok=True)
    print("Directories created successfully.")

    # --- 3. Get and Shuffle Image Files ---
    # We only work with images that have a corresponding label file.
    all_images = sorted([f for f in os.listdir(source_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    
    valid_images = []
    for img in all_images:
        base_name = os.path.splitext(img)[0]
        label_path = os.path.join(source_labels_dir, f"{base_name}.txt")
        if os.path.exists(label_path):
            valid_images.append(img)
        else:
            print(f"[WARNING] Label for image '{img}' not found. Skipping this image.")

    if not valid_images:
        print("[ERROR] No valid images with corresponding labels found. Aborting.")
        sys.exit(1)

    random.seed(42)  # for reproducibility
    random.shuffle(valid_images)
    print(f"Found and shuffled {len(valid_images)} valid images with labels.")

    # --- 4. Calculate Split Index ---
    split_index = int(len(valid_images) * (1 - val_split_ratio))
    train_files = valid_images[:split_index]
    val_files = valid_images[split_index:]

    print(f"Splitting data: {len(train_files)} for training, {len(val_files)} for validation.")

    # --- 5. Function to Copy Files ---
    def copy_files(file_list, dest_img_dir, dest_lbl_dir):
        copied_count = 0
        for filename in file_list:
            base_name, _ = os.path.splitext(filename)
            label_filename = f"{base_name}.txt"

            # Source paths
            src_img_path = os.path.join(source_images_dir, filename)
            src_lbl_path = os.path.join(source_labels_dir, label_filename)

            # Destination paths
            dest_img_path = os.path.join(dest_img_dir, filename)
            dest_lbl_path = os.path.join(dest_lbl_dir, label_filename)

            # Copy files (using copy2 to preserve metadata)
            shutil.copy2(src_img_path, dest_img_path)
            shutil.copy2(src_lbl_path, dest_lbl_path)
            copied_count += 1
        return copied_count

    # --- 6. Execute Copying ---
    print("\nCopying training files...")
    num_train = copy_files(train_files, train_images_dir, train_labels_dir)
    print(f"Copied {num_train} training images and labels.")

    print("\nCopying validation files...")
    num_val = copy_files(val_files, val_images_dir, val_labels_dir)
    print(f"Copied {num_val} validation images and labels.")

    print("\n--- Dataset Split Complete ---")
    print(f"Total files processed: {len(valid_images)}")
    print(f"Output dataset created at: '{output_dir}'")
    print("-" * 30)


if __name__ == '__main__':
    # --- PLEASE CONFIGURE YOUR PATHS HERE ---
    
    # Path to the source dataset directory (containing 'images' and 'labels' folders)
    # Example for Windows: 'C:\\Users\\YourUser\\Desktop\\my_dataset'
    # Example for Linux/Mac: '/home/user/datasets/raw_yolo_data'
    SOURCE_DATASET_DIR ='/home/variphi/gowtham/model_training/airport/New_ariport/'
    # Path where the new split dataset will be created
    OUTPUT_DATASET_DIR ='/home/variphi/gowtham/model_training/airport/New_ariport/split'
    # The ratio for the validation set (e.g., 0.2 means 20%)
    VAL_SPLIT_RATIO = 0.2

    # --- RUN THE SCRIPT ---
    split_yolo_dataset(SOURCE_DATASET_DIR, OUTPUT_DATASET_DIR, VAL_SPLIT_RATIO)

--- Starting YOLO Dataset Split ---
Creating output directories in '/home/variphi/gowtham/model_training/airport/New_ariport/split'...
Directories created successfully.
Found and shuffled 2000 valid images with labels.
Splitting data: 1600 for training, 400 for validation.

Copying training files...
Copied 1600 training images and labels.

Copying validation files...
Copied 400 validation images and labels.

--- Dataset Split Complete ---
Total files processed: 2000
Output dataset created at: '/home/variphi/gowtham/model_training/airport/New_ariport/split'
------------------------------


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/variphi/gowtham/model_training/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/variphi/gowtham/model_training/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/home/variphi/gowtham/model_training/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/variphi/gowtham/model_training/venv/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/variphi/gowtha

: 

In [18]:
import os, glob
from collections import Counter

label_dir = "/home/variphi/gowtham/model_training/dlf_2/split/labels/train/"
class_counts = Counter()

for f in glob.glob(os.path.join(label_dir, "*.txt")):
    for line in open(f):
        parts = line.strip().split()
        if parts:
            class_counts[int(parts[0])] += 1

print("Class distribution:", dict(sorted(class_counts.items())))

Class distribution: {0: 241, 1: 44, 2: 69, 4: 149}


In [13]:
import os

# Paths
image_dir = "/home/variphi/gowtham/albumentation/dataset/no_bolt/project-9-at-2026-05-04-09-53-5f8c87df/images/"
label_dir = "/home/variphi/gowtham/albumentation/dataset/no_bolt/project-9-at-2026-05-04-09-53-5f8c87df/labels/"

# Get sorted file lists
image_files = sorted(os.listdir(image_dir))

# Supported image extensions
valid_ext = [".jpg", ".jpeg", ".png"]

count = 1

for file in image_files:
    name, ext = os.path.splitext(file)

    if ext.lower() in valid_ext:
        # New filename (e.g., img_0001.jpg)
        new_name = f"boltttttt_{count:04d}"
        
        old_image_path = os.path.join(image_dir, file)
        new_image_path = os.path.join(image_dir, new_name + ext)

        # Rename image
        os.rename(old_image_path, new_image_path)

        # Rename corresponding label
        old_label_path = os.path.join(label_dir, name + ".txt")
        new_label_path = os.path.join(label_dir, new_name + ".txt")

        if os.path.exists(old_label_path):
            os.rename(old_label_path, new_label_path)
        else:
            print(f"⚠️ Label not found for {file}")

        count += 1

print("✅ Renaming completed!")

✅ Renaming completed!


In [16]:
import os

# Folder path
image_dir = "/home/variphi/gowtham/albumentation/dataset/bucket_1/output_dataset/images/"   # change this to your folder path

# Prefix to delete
prefix = "boltttttt_"

# Supported image extensions (optional filter)
valid_ext = [".jpg", ".jpeg", ".png"]

deleted_count = 0

for file in os.listdir(image_dir):
    file_path = os.path.join(image_dir, file)

    # Check prefix and extension
    if file.startswith(prefix) and os.path.splitext(file)[1].lower() in valid_ext:
        os.remove(file_path)
        deleted_count += 1
        print(f"Deleted: {file}")

print(f"\n✅ Total files deleted: {deleted_count}")

Deleted: boltttttt_0010__02_AtmosphericFog.jpg
Deleted: boltttttt_0009__01_AdditiveNoise.jpg
Deleted: boltttttt_0059__19_DepthFog.jpg
Deleted: boltttttt_0035__04_CLAHE.jpg
Deleted: boltttttt_0049__12_PlasmaShadow.jpg
Deleted: boltttttt_0041__11_PlasmaBrightnessContrast.jpg
Deleted: boltttttt_0007__17_ToRGB.jpg
Deleted: boltttttt_0030__15_RandomGamma.jpg
Deleted: boltttttt_0025__21_SteamRisingEffect.jpg
Deleted: boltttttt_0053__05_ColorJitter.jpg
Deleted: boltttttt_0063__06_Equalize.jpg
Deleted: boltttttt_0045__08_Illumination.jpg
Deleted: boltttttt_0057__22_DustStorm.jpg
Deleted: boltttttt_0067__23_SandstormEffect.jpg
Deleted: boltttttt_0001__15_RandomGamma.jpg
Deleted: boltttttt_0034__07_FancyPCA.jpg
Deleted: boltttttt_0064__24_AtmosphericHazeGradient.jpg
Deleted: boltttttt_0016__12_PlasmaShadow.jpg
Deleted: boltttttt_0058__26_SmokeDrift.jpg
Deleted: boltttttt_0063__12_PlasmaShadow.jpg
Deleted: boltttttt_0036__08_Illumination.jpg
Deleted: boltttttt_0043__20_Rain.jpg
Deleted: boltttttt

In [ ]:
import os
import shutil
import random

# =========================
# INPUT PATHS
# =========================
image_dir = "/home/variphi/gowtham/albumentation/dataset/04-05-26_bucket/images/"   # <-- change this
label_dir = "/home/variphi/gowtham/albumentation/dataset/04-05-26_bucket/labels/"   # <-- change this

# =========================
# OUTPUT PATH
# =========================
output_dir = "class_3_dataset"

# SETTINGS
target_class = 2
split_ratio = 0.8   # 80% train, 20% val

valid_ext = [".jpg", ".jpeg", ".png"]

# Create folders
for split in ["train", "val"]:
    os.makedirs(os.path.join(output_dir, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, split, "labels"), exist_ok=True)

# =========================
# STEP 1: Collect valid files
# =========================
valid_files = []

for file in os.listdir(image_dir):
    name, ext = os.path.splitext(file)

    if ext.lower() not in valid_ext:
        continue

    lbl_path = os.path.join(label_dir, name + ".txt")
    if not os.path.exists(lbl_path):
        continue

    # Check if class exists
    with open(lbl_path, "r") as f:
        lines = f.readlines()

    filtered = [line for line in lines if int(line.split()[0]) == target_class]

    if len(filtered) > 0:
        valid_files.append((file, filtered))

print(f"✅ Found {len(valid_files)} images with class {target_class}")

# =========================
# STEP 2: Shuffle & Split
# =========================
random.shuffle(valid_files)

split_index = int(len(valid_files) * split_ratio)

train_files = valid_files[:split_index]
val_files = valid_files[split_index:]

print(f"Train: {len(train_files)} | Val: {len(val_files)}")

# =========================
# STEP 3: Copy files
# =========================
def save_files(file_list, split):
    for file, filtered_lines in file_list:
        name, ext = os.path.splitext(file)

        src_img = os.path.join(image_dir, file)
        dst_img = os.path.join(output_dir, split, "images", file)

        src_lbl = os.path.join(label_dir, name + ".txt")
        dst_lbl = os.path.join(output_dir, split, "labels", name + ".txt")

        # Copy image
        shutil.copy(src_img, dst_img)

        # Save filtered label
        with open(dst_lbl, "w") as f:
            f.writelines(filtered_lines)

save_files(train_files, "train")
save_files(val_files, "val")

print("🎯 Done! Dataset ready.")

✅ Found 500 images with class 3
Train: 400 | Val: 100
🎯 Done! Dataset ready.


removing class

In [9]:
import os

# ======================================================
# CONFIGURATION
# ======================================================

LABELS_DIR = "/home/variphi/gowtham/model_training/cumi/part_3/part_3/labels/"   # change this path
CLASS_TO_REMOVE = 8             # class number to remove

# ======================================================
# REMOVE CLASS FROM LABELS
# ======================================================

total_removed = 0
files_processed = 0

for filename in os.listdir(LABELS_DIR):
    if filename.endswith(".txt"):

        file_path = os.path.join(LABELS_DIR, filename)

        with open(file_path, "r") as f:
            lines = f.readlines()

        updated_lines = []
        removed_count = 0

        for line in lines:
            parts = line.strip().split()

            # Skip empty lines
            if len(parts) == 0:
                continue

            class_id = int(parts[0])

            # Remove target class
            if class_id == CLASS_TO_REMOVE:
                removed_count += 1
            else:
                updated_lines.append(line)

        # Rewrite file
        with open(file_path, "w") as f:
            f.writelines(updated_lines)

        files_processed += 1
        total_removed += removed_count

        if removed_count > 0:
            print(f"Removed {removed_count} labels from: {filename}")

print("\n========== DONE ==========")
print(f"Files processed: {files_processed}")
print(f"Total labels removed: {total_removed}")

Removed 1 labels from: 192d5a82__cam163-20260511-101340.txt
Removed 1 labels from: 3279b9ec__cam157-20260512-092412.txt
Removed 2 labels from: e6676364__cam164-20260518-050908.txt
Removed 2 labels from: 1b84e36a__cam164-20260511-104740.txt
Removed 1 labels from: 8b33a1d5__cam163_2bd79738bbab_seg0002_0013.txt
Removed 2 labels from: 961f9f5f__cam164-20260515-065410.txt
Removed 1 labels from: 966f4d38__cam163-20260518-064607.txt
Removed 1 labels from: a9cb1579__cam157-20260515-070811.txt
Removed 1 labels from: 4dc25ea0__cam163_2bd79738bbab_seg0005_0063.txt
Removed 1 labels from: 00a1ea83__cam163-20260511-101440.txt
Removed 1 labels from: 70b7a3ce__cam164_b6c98ea9ca75_seg0002_0003.txt
Removed 1 labels from: 119de5ed__cam163-20260511-100040.txt
Removed 2 labels from: 30131ab0__cam164-20260518-062308.txt
Removed 1 labels from: 4178aabe__cam164_96299b2bc251_seg0003_0020.txt
Removed 1 labels from: f6aad7e1__cam164-20260516-083709.txt
Removed 2 labels from: e8371cae__cam164-20260516-064409.txt
